In [10]:
import pandas as pd
import numpy as np

In [11]:
FILE_PRECLINICAL_LINKING = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_preclin.csv"
FILE_CLINICAL_LINKING = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_clin.csv"
FILE_CLINICAL_METADATA = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/06_preclin_clinic_join/data/clinical/clinical_nct_docs_metadata_20240313.csv"
PRECLINICAL_METADATA_PATH = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/02_animal_study_classification/data/animal_studies/full_pubmed_filtered_animal_6002827_metadata.csv" #"/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/03_IE_ner/data/animal_studies_with_drug_disease/animal_studies_metadata_595768.csv"

OUTPUT_DIR = "06_preclin_clinic_join/data/joined_data/"

conditions_col_to_use = "merged_mondo_label" # #"linkbert_mapped_conditions" # disease_term_mondo_norm
drugs_col_to_use =  "merged_umls_label" #"linkbert_mapped_drugs" #"drug_term_umls_norm"

conditions_col_to_use_clinical =  "merged_mondo_label"#"linkbert_aact_mapped_conditions" #"disease_term_mondo_norm"
drugs_col_to_use_clinical =  "merged_umls_label" #"linkbert_aact_mapped_drugs" # "drug_term_umls_norm"


In [12]:
def count_unique_from_pipe_column(df, column):
    """
    Count unique items and their frequencies in a DataFrame column containing '|' separated values.

    Returns:
        total_unique (int): total number of unique non-empty terms
        freq_df (pd.DataFrame): columns ['term', 'n_articles']
                               where 'n_articles' = number of unique PMIDs (rows) mentioning that term
    """
    import pandas as pd

    # explode values
    all_items = (
        df[[column, "PMID"]]
        .dropna(subset=[column])
        .assign(**{column: df[column].astype(str).str.split("|")})
        .explode(column)
    )
    all_items[column] = all_items[column].str.strip()
    all_items = all_items[all_items[column] != ""]

    # count how many distinct PMIDs mention each term
    freq = (
        all_items.groupby(column)["PMID"]
        .nunique()
        .reset_index(name="n_articles")
        .sort_values("n_articles", ascending=False)
    )

    total_unique = freq.shape[0]
    return total_unique, freq


## Preclinical Data

In [13]:
# --- Load Preclinical Data ---
preclinical_df = pd.read_csv(FILE_PRECLINICAL_LINKING)
print(f"Shape of preclinical_df: {preclinical_df.shape}, {preclinical_df.PMID.nunique()} unique PMIDs")
metadata_df_year = pd.read_csv(PRECLINICAL_METADATA_PATH)[['PMID','year']]
metadata_df_year = metadata_df_year.drop_duplicates(subset=['PMID'])

preclinical_df = preclinical_df.merge(metadata_df_year, on="PMID", how="left")
preclinical_df = preclinical_df.rename(columns={
    'year': 'pub_year'
})

Shape of preclinical_df: (540999, 14), 540999 unique PMIDs


In [14]:
n_unique, freq = count_unique_from_pipe_column(preclinical_df, "merged_umls_label")
print(f"Unique count: {n_unique}")

Unique count: 292514


In [15]:
freq

,merged_umls_label,n_articles
35534,Dexamethasone,5806
29728,Acetylcysteine,4559
46202,NG-Nitroarginine Methyl Ester,4306
52975,Sirolimus,4107
35945,Doxorubicin,4021
...,...,...
118778,dbmib,1
118779,dbo 17 and dbo 11,1
118780,dbpr807,1
118781,dbpt,1


In [16]:
# Split and explode conditions and drugs
#preclinical_df[conditions_col_to_use] = preclinical_df[conditions_col_to_use].str.split("|")
#preclinical_df = preclinical_df.explode(conditions_col_to_use, ignore_index=True)

# DISEASE
cols_to_explode = [
    conditions_col_to_use,
    "merged_mondo_termid"
    
]

for col in cols_to_explode:
    preclinical_df[col] = preclinical_df[col].astype(str).str.split("|")

preclinical_df = preclinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique disease before length filter: {preclinical_df[conditions_col_to_use].nunique()}")
#preclinical_df = preclinical_df[preclinical_df[conditions_col_to_use].fillna("").str.len() > 2]
#print(f"Unique disease: {preclinical_df[conditions_col_to_use].nunique()}")

# DRUG
cols_to_explode = [
    drugs_col_to_use,     # e.g. drug names
    "merged_umls_termid",             # IDs
    
]

for col in cols_to_explode:
    preclinical_df[col] = preclinical_df[col].astype(str).str.split("|")

preclinical_df = preclinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique drugs before length filter: {preclinical_df[drugs_col_to_use].nunique()}")

preclinical_df = preclinical_df[preclinical_df[drugs_col_to_use].fillna("").str.len() > 2]
print(f"Unique drugs: {preclinical_df[drugs_col_to_use].nunique()}")

preclinical_df = preclinical_df.drop_duplicates()

# Strip whitespace and convert to lowercase
preclinical_df[conditions_col_to_use] = preclinical_df[conditions_col_to_use].str.strip().str.lower()
preclinical_df[drugs_col_to_use] = preclinical_df[drugs_col_to_use].str.strip().str.lower()

Unique disease before length filter: 79918
Unique drugs before length filter: 292514
Unique drugs: 291465


In [17]:
preclinical_df['disease<>drug'] = (
    preclinical_df[conditions_col_to_use] + " <> " + preclinical_df[drugs_col_to_use]
)

In [18]:
preclinical_df.head()

,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid,pub_year,disease<>drug
0,31733831,asthma,asthma,MONDO:0004979,asthma,-1,asthma,MONDO:0004979,isorhynchophylline,isorhynchophylline,C0245133,-1,isorhynchophylline,C0245133,2020.0,asthma <> isorhynchophylline
1,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,"mirn1192 microrna, mouse",C3849422,2020.0,"myocardial infarction <> mirn1192 microrna, mouse"
2,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,antgomir-1192,-1,2020.0,myocardial infarction <> antgomir-1192
3,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,agomir-1192,-1,2020.0,myocardial infarction <> agomir-1192
4,31733925,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,-1,systemic lupus erythematosus,MONDO:0007915,hla-g2|g2,HLA-G2 Isoform|g2,C0967254|-1,-1,hla-g2 isoform,C0967254,2020.0,systemic lupus erythematosus <> hla-g2 isoform


In [19]:
mapping_lookup_preclin_drug = dict(
    zip(
        preclinical_df["merged_umls_label"],
        preclinical_df["merged_umls_termid"]
    )
)

mapping_lookup_preclin_disease = dict(
    zip(
        preclinical_df["merged_mondo_label"],
        preclinical_df["merged_mondo_termid"]
    )
)

## Clinical Data

In [20]:
# --- Load Clinical Data ---
clinical_df = pd.read_csv(FILE_CLINICAL_LINKING)
print(f"Shape of clinical_df: {clinical_df.shape}, {clinical_df.nct_id.nunique()} unique NCTIDs")

# DISEASE
cols_to_explode = [
    conditions_col_to_use_clinical,
    "merged_mondo_termid"
    
]

for col in cols_to_explode:
    clinical_df[col] = clinical_df[col].astype(str).str.split("|")

clinical_df = clinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique disease before length filter: {clinical_df[conditions_col_to_use_clinical].nunique()}")

# DRUG
cols_to_explode = [
    drugs_col_to_use_clinical,     # e.g. drug names
    "merged_umls_termid",             # IDs
]

for col in cols_to_explode:
    clinical_df[col] = clinical_df[col].astype(str).str.split("|")

clinical_df = clinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique drugs before length filter: {clinical_df[drugs_col_to_use_clinical].nunique()}")
clinical_df = clinical_df[clinical_df[drugs_col_to_use_clinical].fillna("").str.len() > 2]
print(f"Unique drugs: {clinical_df[drugs_col_to_use_clinical].nunique()}")

# Strip whitespace and convert to lowercase
clinical_df[conditions_col_to_use_clinical] = clinical_df[conditions_col_to_use_clinical].str.strip().str.lower()
clinical_df[drugs_col_to_use_clinical] = clinical_df[drugs_col_to_use_clinical].str.strip().str.lower()


Shape of clinical_df: (18609, 14), 18609 unique NCTIDs
Unique disease before length filter: 11024
Unique drugs before length filter: 12904
Unique drugs: 12705


In [21]:
# Create disease-drug key
clinical_df['disease<>drug'] = (
    clinical_df[conditions_col_to_use_clinical] + " <> " + clinical_df[drugs_col_to_use_clinical]
)


# Load and merge clinical metadata (phase + status)
metadata_df = pd.read_csv(FILE_CLINICAL_METADATA)[['nct_id', 'phase', 'overall_status', 'start_date', 'completion_date']]
# get start year
metadata_df['start_date'] = pd.to_datetime(metadata_df['start_date'], errors='coerce')
metadata_df['trial_start_year'] = metadata_df['start_date'].dt.year
# get completion year
metadata_df['completion_date'] = pd.to_datetime(metadata_df['completion_date'], errors='coerce')
metadata_df['trial_completion_year'] = metadata_df['completion_date'].dt.year
metadata_df = metadata_df.drop_duplicates()



In [22]:
clinical_df = clinical_df.merge(metadata_df, on='nct_id', how='left')

In [23]:
clinical_df.shape

(124290, 21)

In [24]:
clinical_df.head()

,nct_id,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,...,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid,disease<>drug,phase,overall_status,start_date,completion_date,trial_start_year,trial_completion_year
0,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicidal,-1,ketamine,Ketamine,...,-1,ketamine,C0022614,suicidal <> ketamine,Phase 2,Withdrawn,2019-04-01,2019-06-30,2019.0,2019.0
1,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicidal ideation,-1,ketamine,Ketamine,...,-1,ketamine,C0022614,suicidal ideation <> ketamine,Phase 2,Withdrawn,2019-04-01,2019-06-30,2019.0,2019.0
2,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicide,-1,ketamine,Ketamine,...,-1,ketamine,C0022614,suicide <> ketamine,Phase 2,Withdrawn,2019-04-01,2019-06-30,2019.0,2019.0
3,NCT05216770,laryngeal dystonia|voice tremor|Tremor,spasmodic dystonia|voice tremor|obsolete rare ...,MONDO:0000485|-1|MONDO:0017644,spasmodic dystonia|voice tremor|obsolete rare ...,focal dystonia|-1,spasmodic dystonia,MONDO:0000485,Laryngeal sensory block with topical bupivacaine,Laryngeal sensory block with topical bupivacaine,...,-1,laryngeal sensory block with topical bupivacaine,-1,spasmodic dystonia <> laryngeal sensory block ...,Early Phase 1,Recruiting,2022-03-24,2026-08-31,2022.0,2026.0
4,NCT05216770,laryngeal dystonia|voice tremor|Tremor,spasmodic dystonia|voice tremor|obsolete rare ...,MONDO:0000485|-1|MONDO:0017644,spasmodic dystonia|voice tremor|obsolete rare ...,focal dystonia|-1,voice tremor,-1,Laryngeal sensory block with topical bupivacaine,Laryngeal sensory block with topical bupivacaine,...,-1,laryngeal sensory block with topical bupivacaine,-1,voice tremor <> laryngeal sensory block with t...,Early Phase 1,Recruiting,2022-03-24,2026-08-31,2022.0,2026.0


## Aggregate for merging

In [25]:
clinical_doc_id_col = "nct_id"
clinical_key_col = 'disease<>drug'

In [26]:
clinical_unique = (
        clinical_df
        .sort_values(clinical_doc_id_col)  # optional: enforce a deterministic order
        .drop_duplicates(subset=[clinical_key_col, clinical_doc_id_col], keep='first')
    )
# === Aggregate clinical data by key ===
clinical_agg = (
        clinical_unique
        .groupby(clinical_key_col)
        .agg({
            clinical_doc_id_col: list,
            'merged_mondo_termid': 'first',
            'merged_umls_termid': 'first',
            'merged_mondo_label': 'first',
            'merged_umls_label': 'first',
            'trial_start_year':     list,
            'trial_completion_year':     list,
            'phase':              list,
            'overall_status':     list
        })
        .reset_index()
        )
    

# Rename columns for clarity and consistency
clinical_agg.rename(columns={
    clinical_key_col: 'normalized_key',
    clinical_doc_id_col: 'clinical_doc_ids'
}, inplace=True)

# Add count of clinical documents per key
clinical_agg['clinical_count'] = clinical_agg['clinical_doc_ids'].apply(len)

In [27]:
clinical_agg[clinical_agg['merged_mondo_termid']!="-1"]

,normalized_key,clinical_doc_ids,merged_mondo_termid,merged_umls_termid,merged_mondo_label,merged_umls_label,trial_start_year,trial_completion_year,phase,overall_status,clinical_count
431,22q11.2 deletion syndrome <> antihypertensive,[NCT01423396],MONDO:0018923,-1,22q11.2 deletion syndrome,antihypertensive,[2010.0],[2022.0],[Not Applicable],[Completed],1
463,7q11.23 microduplication syndrome <> physical ...,[NCT04051086],MONDO:0012342,-1,7q11.23 microduplication syndrome,physical examination and urine and blood samples,[2019.0],[2021.0],[Not Applicable],[Unknown status],1
499,"abdominal obesity-metabolic syndrome <> 2,3-bi...",[NCT01234506],MONDO:0000816,C0045341,abdominal obesity-metabolic syndrome,"2,3-bis(3'-hydroxybenzyl)butyrolactone",[2010.0],[2013.0],[Phase 2],[Completed],1
500,abdominal obesity-metabolic syndrome <> 24 hr ...,[NCT00266864],MONDO:0000816,C0980517,abdominal obesity-metabolic syndrome,24 hr testosterone 0.208 mg/hr transdermal system,[2003.0],[2012.0],[Phase 2/Phase 3],[Completed],1
501,abdominal obesity-metabolic syndrome <> 5-fluo...,[NCT04898270],MONDO:0000816,C0049194,abdominal obesity-metabolic syndrome,5-fluorouridine 5'-triphosphate,[2019.0],[2021.0],[Phase 4],[Completed],1
...,...,...,...,...,...,...,...,...,...,...,...
85483,young-onset parkinson disease <> venglustat,[NCT02906020],MONDO:0017279,C4726953,young-onset parkinson disease,venglustat,[2016.0],[2021.0],[Phase 2],[Terminated],1
85484,young-onset parkinson disease <> venglustat gz...,[NCT02906020],MONDO:0017279,-1,young-onset parkinson disease,venglustat gz/sar402671,[2016.0],[2021.0],[Phase 2],[Terminated],1
85485,young-onset parkinson disease <> win-1001x,[NCT04220762],MONDO:0017279,-1,young-onset parkinson disease,win-1001x,[2020.0],[2021.0],[Phase 2],[Unknown status],1
85486,young-onset parkinson disease <> zinpentraxin ...,[NCT05924243],MONDO:0017279,C5667141,young-onset parkinson disease,zinpentraxin alfa,[2022.0],[2025.0],[Phase 1],[Recruiting],1


In [28]:
clinical_pairs = set(clinical_agg['normalized_key'])

In [29]:
preclinical_key_col = 'disease<>drug'
preclinical_doc_id_col = "PMID"
preclinical_disease_col = conditions_col_to_use

In [30]:
# === Aggregate preclinical data by key ===
preclinical_agg = preclinical_df.groupby(preclinical_key_col).agg({
    preclinical_doc_id_col: list,
    'pub_year':list,
    'merged_mondo_termid': 'first',
    'merged_umls_termid': 'first',
}).reset_index()

# Rename columns for consistency
preclinical_agg.rename(columns={
    preclinical_key_col: 'normalized_key',
    preclinical_doc_id_col: 'preclinical_doc_ids',
}, inplace=True)

# Add count of preclinical documents per key
preclinical_agg['preclinical_count'] = preclinical_agg['preclinical_doc_ids'].apply(len)

In [31]:
preclinical_agg.head()

,normalized_key,preclinical_doc_ids,pub_year,merged_mondo_termid,merged_umls_termid,preclinical_count
0,""" hemorrhagic shock <> saline",[1742862],[1991.0],-1,C0036082,1
1,""" indium lung <> acid, edetic",[34229798],[2021.0],-1,C0013618,1
2,""" ischemic hepatitis <> allopurinol",[9156787],[1996.0],-1,C0002144,1
3,""" nitrozation stress <> s-(4-quinazolyl) merca...",[15341075],[2004.0],-1,-1,1
4,""" stunning <> alpha tocopherol analogue",[8143306],[1994.0],-1,-1,1


In [32]:
preclinical_pairs = set(preclinical_agg['normalized_key'])

## Merge clinical and preclinical on disease <> drug

In [33]:
merged_df = pd.merge(clinical_agg, preclinical_agg, on='normalized_key', how='outer', suffixes=("", "_right"))

In [34]:
def clean_and_sort_study_data(df):
    """
    Cleans and sorts a merged clinical/preclinical DataFrame by removing invalid rows
    and ordering by study counts.

    Specifically:
    - Sorts the DataFrame in descending order of 'clinical_count' and 'preclinical_count'.
    - Removes rows where either 'clinical_count' or 'preclinical_count' is NaN.
    - Removes rows where the 'drug' name is missing or has 3 or fewer characters.
    - Prints the shape before and after filtering for transparency.

    Parameters:
    ----------
    df : pd.DataFrame
        A DataFrame containing 'clinical_count', 'preclinical_count', and 'drug'.

    Returns:
    -------
    pd.DataFrame
        The cleaned and sorted DataFrame.
    """
    print(f'Input shape of merged for cleaning: {df.shape}')

    # Sort by descending clinical and preclinical study counts
    sorted_df = df.sort_values(
        by=['clinical_count', 'preclinical_count'],
        ascending=[False, False]
    )

    # Remove rows with missing study counts
    filtered_df = sorted_df.dropna(subset=['clinical_count', 'preclinical_count'])
    print(f'Shape after filtering NaN counts: {filtered_df.shape}')

    # Remove rows with invalid or too-short drug names
    #filtered_df = filtered_df[
    #    filtered_df['drug'].apply(lambda x: isinstance(x, str) and len(x.strip()) > 3)
    #]
    #print(f'Shape after filtering short or invalid drugs: {filtered_df.shape}')

    print(f'Shape after filtering empty counts and short drugs: {filtered_df.shape}')
    return filtered_df

In [35]:
merged_df.head()

,normalized_key,clinical_doc_ids,merged_mondo_termid,merged_umls_termid,merged_mondo_label,merged_umls_label,trial_start_year,trial_completion_year,phase,overall_status,clinical_count,preclinical_doc_ids,pub_year,merged_mondo_termid_right,merged_umls_termid_right,preclinical_count
0,""" <> apomorphine",[NCT02542696],-1,C0003596,"""",apomorphine,[2015.0],[2022.0],[Phase 3],[Completed],1.0,NaN,NaN,NaN,NaN,NaN
1,""" <> evt 301",[NCT01617135],-1,C2715494,"""",evt 301,[2012.0],[2012.0],[Phase 2],[Completed],1.0,NaN,NaN,NaN,NaN,NaN
2,""" <> interferon alpha-2",[NCT00846430],-1,C2937361,"""",interferon alpha-2,[2008.0],[2017.0],[Phase 2],[Completed],1.0,NaN,NaN,NaN,NaN,NaN
3,""" <> levodopa",[NCT01617135],-1,C0023570,"""",levodopa,[2012.0],[2012.0],[Phase 2],[Completed],1.0,NaN,NaN,NaN,NaN,NaN
4,""" <> peginterferon alfa-2b",[NCT00846430],-1,C0796545,"""",peginterferon alfa-2b,[2008.0],[2017.0],[Phase 2],[Completed],1.0,NaN,NaN,NaN,NaN,NaN


In [36]:
filtered_df = clean_and_sort_study_data(merged_df)
filtered_df.head()

Input shape of merged for cleaning: (1879982, 16)
Shape after filtering NaN counts: (13570, 16)
Shape after filtering empty counts and short drugs: (13570, 16)


,normalized_key,clinical_doc_ids,merged_mondo_termid,merged_umls_termid,merged_mondo_label,merged_umls_label,trial_start_year,trial_completion_year,phase,overall_status,clinical_count,preclinical_doc_ids,pub_year,merged_mondo_termid_right,merged_umls_termid_right,preclinical_count
60139,parkinson disease <> levodopa,"[NCT00004576, NCT00006077, NCT00006337, NCT000...",MONDO:0005180,C0023570,parkinson disease,levodopa,"[2000.0, 2000.0, 2000.0, 2001.0, 2001.0, 2002....","[2000.0, 2003.0, 2002.0, 2003.0, 2005.0, 2005....","[Phase 2, Phase 2, Phase 2, Phase 2, Phase 2, ...","[Completed, Completed, Completed, Completed, C...",252.0,"[31915844, 31920431, 11860157, 11860478, 11860...","[2020.0, 2020.0, 2002.0, 2001.0, 2002.0, 2010....",MONDO:0005180,C0023570,1737.0
72105,schizophrenia <> risperidone,"[NCT00014001, NCT00018668, NCT00034749, NCT000...",MONDO:0005090,C0073393,schizophrenia,risperidone,"[2000.0, 2000.0, 2001.0, 2001.0, 2002.0, 2001....","[2004.0, 2004.0, 2006.0, 2002.0, 2005.0, 2002....","[Phase 4, Phase 4, Phase 3, Phase 3, Phase 3, ...","[Completed, Completed, Completed, Completed, C...",221.0,"[32866524, 24317442, 36744005, 19579000, 27250...","[2020.0, 2014.0, 2023.0, 2010.0, 2016.0, 2001....",MONDO:0005090,C0073393,454.0
71930,schizophrenia <> olanzapine,"[NCT00014001, NCT00034801, NCT00034892, NCT000...",MONDO:0005090,C0171023,schizophrenia,olanzapine,"[2000.0, 2001.0, 2002.0, 2001.0, 2001.0, 2001....","[2004.0, 2003.0, 2005.0, 2002.0, 2003.0, 2006....","[Phase 4, Phase 4, Phase 3, Phase 4, Phase 4, ...","[Completed, Completed, Completed, Completed, C...",162.0,"[32866524, 19359144, 15464760, 19059234, 28375...","[2020.0, 2009.0, 2004.0, 2009.0, 2017.0, 2014....",MONDO:0005090,C0171023,426.0
71330,schizophrenia <> aripiprazole,"[NCT00036127, NCT00036361, NCT00080327, NCT000...",MONDO:0005090,C0299792,schizophrenia,aripiprazole,"[2002.0, 2002.0, 2003.0, 2003.0, 2004.0, 2004....","[2003.0, 2003.0, 2004.0, 2007.0, 2006.0, 2005....","[Phase 2/Phase 3, Phase 3, Phase 4, Phase 4, P...","[Completed, Completed, Completed, Completed, C...",138.0,"[24317442, 24309096, 21893111, 17056032, 34332...","[2014.0, 2014.0, 2011.0, 2006.0, 2021.0, 2006....",MONDO:0005090,C0299792,196.0
21981,depressive disorder <> antidepressant,"[NCT00009191, NCT00018902, NCT00026637, NCT000...",MONDO:0002050,-1,depressive disorder,antidepressant,"[2005.0, 2001.0, 2001.0, 2002.0, 2004.0, 2004....","[2007.0, 2007.0, 2008.0, 2006.0, 2009.0, 2014....","[Phase 4, Phase 2/Phase 3, Phase 3, Phase 4, P...","[Completed, Completed, Completed, Completed, C...",131.0,"[29157077, 22983097, 33245959, 38331354, 31856...","[2018.0, 2013.0, 2021.0, 2024.0, 2019.0, 2003....",MONDO:0002050,-1,65.0


In [37]:
phase_order = {
    'Early Phase 1': 0,
    'Phase 1': 1,
    'Phase 1/Phase 2': 1.5,
    'Phase 2': 2,
    'Phase 2/Phase 3': 2.5,
    'Phase 3': 3,
    'Phase 4': 4,
    'Not Applicable': -1  # Lowest value to ignore in max comparison
}

# Get max phase for each row
def get_max_phase(phases):
    max_val = -1
    max_phase = 'Not Applicable'
    for phase in phases:
        val = phase_order.get(phase, -1)
        if val > max_val:
            max_val = val
            max_phase = phase
    return max_phase

def first_completed_phase3_completion_year(phases, statuses, completion_years):
    """
    Return the earliest completion year among trials that are Phase 3 and Completed.
    Returns None if not available / no matches.
    """
    if not all(isinstance(x, list) for x in [phases, statuses, completion_years]):
        return None

    years = []
    for p, s, y in zip(phases, statuses, completion_years):
        if ("Phase 3" in str(p)) and (str(s).strip().lower() == "completed"):
            try:
                yy = int(y) if y is not None and str(y).strip() != "" else None
            except Exception:
                yy = None
            if yy is not None:
                years.append(yy)

    return min(years) if years else None



In [38]:
filtered_df['max_phase'] = filtered_df['phase'].apply(get_max_phase)
filtered_df['min_trial_start_year'] = filtered_df['trial_start_year'].apply(
    lambda x: min(x) if isinstance(x, list) and len(x) > 0 else None
)

filtered_df['max_trial_start_year'] = filtered_df['trial_start_year'].apply(
    lambda x: max(x) if isinstance(x, list) and len(x) > 0 else None
)

filtered_df["min_phase_4_year"] = filtered_df.apply(
    lambda row: min(
        [year for phase, year in zip(row["phase"], row["trial_start_year"]) if "Phase 4" in str(phase)], 
        default=None
    ),
    axis=1
)

# Identify studies with Phase 3 or 4 trials
filtered_df['at_least_one_phase3'] = filtered_df['phase'].apply(
    lambda phases: any("Phase 3" in str(p) for p in phases)
)

filtered_df["at_least_one_phase3_completed"] = filtered_df.apply(
    lambda row: any(
        ("Phase 3" in str(p)) and (str(s) == "Completed")
        for p, s in zip(row["phase"], row["overall_status"])
    ),
    axis=1
)

filtered_df['at_least_one_phase4'] = filtered_df['phase'].apply(
    lambda phases: any("Phase 4" in str(p) for p in phases)
)

filtered_df["trial_completion_year_first_completed_phase3"] = filtered_df.apply(
    lambda row: first_completed_phase3_completion_year(
        row["phase"],
        row["overall_status"],
        row["trial_completion_year"],
    ),
    axis=1
)

In [39]:
filtered_df = filtered_df.drop(columns=['merged_umls_termid_right', 'merged_mondo_termid_right'])

## Add FDA info

In [40]:
fda_drug_disease = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drug_disease_pairs_mapped_both.csv")


In [41]:
fda_drug_disease['disease<>drug'] = (
    fda_drug_disease["disease_mondo_term_norm"] + " <> " + fda_drug_disease["merged_umls_label"]
)
fda_drug_disease = fda_drug_disease.add_prefix("fda_")
fda_drug_disease['fda_AP']=True
fda_drug_disease.head()

,fda_merged_umls_label,fda_disease_mondo_term_norm,fda_earliest_year,fda_disease,fda_documents,fda_disease<>drug,fda_AP
0,(+)-Tolterodine,frequency,1998.0,frequency,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",frequency <> (+)-Tolterodine,True
1,(+)-Tolterodine,overactive bladder,1998.0,overactive bladder,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",overactive bladder <> (+)-Tolterodine,True
2,(+)-Tolterodine,urge urinary incontinence,1998.0,urge urinary incontinence,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",urge urinary incontinence <> (+)-Tolterodine,True
3,(+)-Tolterodine,urgency,1998.0,urgency,"['ANDA077006', 'ANDA079141', 'ANDA203016', 'AN...",urgency <> (+)-Tolterodine,True
4,(+-)-Milnacipran,fibromyalgia,2009.0,fibromyalgia,"['ANDA205081', 'NDA022256']",fibromyalgia <> (+-)-Milnacipran,True


In [42]:
fda_drug_disease[
    fda_drug_disease['fda_merged_umls_label']
    .str.contains("metformin", case=False, na=False)
]


,fda_merged_umls_label,fda_disease_mondo_term_norm,fda_earliest_year,fda_disease,fda_documents,fda_disease<>drug,fda_AP
3725,"Hydrochloride, Metformin",diabetic ketoacidosis,2015.0,diabetic ketoacidosis,['ANDA204802'],"diabetic ketoacidosis <> Hydrochloride, Metformin",True
3726,"Hydrochloride, Metformin",high blood sugar,2008.0,high blood sugar,['ANDA078321'],"high blood sugar <> Hydrochloride, Metformin",True
3727,"Hydrochloride, Metformin",type 1 diabetes mellitus,2015.0,type 1 diabetes mellitus,['ANDA204802'],"type 1 diabetes mellitus <> Hydrochloride, Met...",True
3728,"Hydrochloride, Metformin",type 2 diabetes mellitus,2002.0,type 2 diabetes mellitus,"['ANDA075965', 'ANDA075972', 'ANDA075973', 'AN...","type 2 diabetes mellitus <> Hydrochloride, Met...",True
4889,Metformin,diabetic ketoacidosis,2015.0,diabetic ketoacidosis,['ANDA204802'],diabetic ketoacidosis <> Metformin,True
4890,Metformin,high blood sugar,2008.0,high blood sugar,['ANDA078321'],high blood sugar <> Metformin,True
4891,Metformin,type 1 diabetes mellitus,2015.0,type 1 diabetes mellitus,['ANDA204802'],type 1 diabetes mellitus <> Metformin,True
4892,Metformin,type 2 diabetes mellitus,2002.0,type 2 diabetes mellitus,"['ANDA075965', 'ANDA075972', 'ANDA075973', 'AN...",type 2 diabetes mellitus <> Metformin,True


In [43]:
fda_pairs = set(fda_drug_disease['fda_disease<>drug'])

In [44]:
len(fda_pairs)

9350

In [45]:
# pairwise overlaps
fda_drugs = {str(x).strip().lower() for x in fda_pairs if x}
clinical_drugs = {str(x).strip().lower() for x in clinical_pairs if x}
preclinical_drugs = {str(x).strip().lower() for x in preclinical_pairs if x}

fda_clinical = fda_drugs & clinical_drugs
fda_preclinical = fda_drugs & preclinical_drugs
clinical_preclinical = clinical_drugs & preclinical_drugs

# all three
all_three = fda_drugs & clinical_drugs & preclinical_drugs

print("FDA ∩ Clinical:", len(fda_clinical))
print("FDA ∩ Preclinical:", len(fda_preclinical))
print("Clinical ∩ Preclinical:", len(clinical_preclinical))
print("All three:", len(all_three))


FDA ∩ Clinical: 1017
FDA ∩ Preclinical: 3068
Clinical ∩ Preclinical: 13570
All three: 815


In [46]:
fda_preclin_not_clin = (fda_drugs & preclinical_drugs) - clinical_drugs
len(fda_preclin_not_clin)

2253

In [47]:
fda_drug_disease["fda_disease<>drug"] = (
    fda_drug_disease["fda_disease<>drug"].astype(str).str.strip().str.lower()
)
filtered_df["normalized_key"] = (
    filtered_df["normalized_key"].astype(str).str.strip().str.lower()
)
filtered_df = filtered_df.merge(
    fda_drug_disease,
    how="left",
    left_on="normalized_key",
    right_on="fda_disease<>drug"
)
filtered_df["fda_AP"] = filtered_df["fda_AP"].fillna(False).astype(bool)


In [48]:
filtered_df["fda_AP"].value_counts()


fda_AP
False    12755
True       815
Name: count, dtype: int64

#### save for association analysis

In [49]:
filtered_df.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease.csv", index=False) # used in Translation_05!
filtered_df.to_csv("./out/translation_table_drug_disease.csv", index=False) # used in Translation_05!

In [50]:
filtered_df[filtered_df["fda_earliest_year"].isna() & filtered_df['at_least_one_phase4']][['normalized_key', 'clinical_doc_ids', 'merged_mondo_termid',
       'merged_umls_termid', 'merged_mondo_label', 'merged_umls_label',
       'trial_start_year', 'phase', 'overall_status', 'clinical_count',
       'preclinical_doc_ids', 'pub_year', 'preclinical_count','min_phase_4_year', 'trial_completion_year_first_completed_phase3']]#.head(10)

,normalized_key,clinical_doc_ids,merged_mondo_termid,merged_umls_termid,merged_mondo_label,merged_umls_label,trial_start_year,phase,overall_status,clinical_count,preclinical_doc_ids,pub_year,preclinical_count,min_phase_4_year,trial_completion_year_first_completed_phase3
4,depressive disorder <> antidepressant,"[NCT00009191, NCT00018902, NCT00026637, NCT000...",MONDO:0002050,-1,depressive disorder,antidepressant,"[2005.0, 2001.0, 2001.0, 2002.0, 2004.0, 2004....","[Phase 4, Phase 2/Phase 3, Phase 3, Phase 4, P...","[Completed, Completed, Completed, Completed, C...",131.0,"[29157077, 22983097, 33245959, 38331354, 31856...","[2018.0, 2013.0, 2021.0, 2024.0, 2019.0, 2003....",65.0,2001.0,2004.0
6,drug dependence <> nicotine,"[NCT00046813, NCT00108537, NCT00124683, NCT001...",MONDO:0005303,C0028040,drug dependence,nicotine,"[2001.0, 2003.0, 2001.0, 2002.0, 2001.0, 2002....","[Phase 4, Phase 3, Phase 2, Phase 2, Phase 4, ...","[Terminated, Completed, Completed, Completed, ...",123.0,"[29472642, 24304823, 30035805, 37703771, 22178...","[2018.0, 2014.0, 2018.0, 2023.0, 2012.0, 2006....",47.0,2001.0,2005.0
7,nicotine dependence <> nicotine,"[NCT00046813, NCT00108537, NCT00124683, NCT001...",MONDO:0008575,C0028040,nicotine dependence,nicotine,"[2001.0, 2003.0, 2001.0, 2002.0, 2001.0, 2002....","[Phase 4, Phase 3, Phase 2, Phase 2, Phase 4, ...","[Terminated, Completed, Completed, Completed, ...",121.0,"[29472642, 24304823, 30035805, 17046158, 30293...","[2018.0, 2014.0, 2018.0, 2006.0, 2018.0, 2024....",32.0,2001.0,2005.0
8,depressive disorder <> ketamine,"[NCT00088699, NCT00749203, NCT01046630, NCT012...",MONDO:0002050,C0022614,depressive disorder,ketamine,"[2004.0, 2009.0, 2009.0, 2010.0, 2011.0, 2011....","[Phase 1/Phase 2, Phase 2, Phase 1, Phase 1, P...","[Completed, Completed, Completed, Completed, C...",118.0,"[31187076, 26647181, 27066531, 26751284, 26660...","[2019.0, 2015.0, 2016.0, 2016.0, 2016.0, 2022....",587.0,2010.0,2021.0
11,depressive disorder <> escitalopram,"[NCT00070694, NCT00071643, NCT00073411, NCT000...",MONDO:0002050,C1099456,depressive disorder,escitalopram,"[2003.0, 2002.0, 2003.0, 2003.0, 2005.0, 2005....","[Phase 2, Not Applicable, Phase 3, Phase 4, No...","[Completed, Completed, Completed, Completed, C...",98.0,"[34299103, 35733802, 30658246, 20599917, 34016...","[2021.0, 2022.0, 2019.0, 2010.0, 2021.0, 2021....",244.0,2002.0,2005.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13529,vascular brain injury <> glycerylphosphorylcho...,[NCT02648906],MONDO:0005621,C0017889,vascular brain injury,glycerylphosphorylcholine,[2015.0],[Phase 4],[Unknown status],1.0,[16989788],[2006.0],1.0,2015.0,NaN
13530,vascular brain injury <> melatonin,[NCT03115034],MONDO:0005621,C0025219,vascular brain injury,melatonin,[2016.0],[Phase 4],[Completed],1.0,[33839695],[2021.0],1.0,2016.0,NaN
13533,vascular dementia <> escitalopram,[NCT00229333],MONDO:0004648,C1099456,vascular dementia,escitalopram,[2004.0],[Phase 4],[Unknown status],1.0,[27966748],[2016.0],1.0,2004.0,NaN
13545,visceral pain <> bupivacaine,[NCT05792774],-1,C0006400,visceral pain,bupivacaine,[2023.0],[Phase 4],[Completed],1.0,[34347648],[2021.0],1.0,2023.0,NaN


In [51]:
filtered_df.shape

(13570, 29)

In [52]:
filtered_df.merged_mondo_label.nunique(), filtered_df.merged_umls_label.nunique()

(1383, 2818)

In [53]:
filtered_df.columns

Index(['normalized_key', 'clinical_doc_ids', 'merged_mondo_termid',
       'merged_umls_termid', 'merged_mondo_label', 'merged_umls_label',
       'trial_start_year', 'trial_completion_year', 'phase', 'overall_status',
       'clinical_count', 'preclinical_doc_ids', 'pub_year',
       'preclinical_count', 'max_phase', 'min_trial_start_year',
       'max_trial_start_year', 'min_phase_4_year', 'at_least_one_phase3',
       'at_least_one_phase3_completed', 'at_least_one_phase4',
       'trial_completion_year_first_completed_phase3', 'fda_merged_umls_label',
       'fda_disease_mondo_term_norm', 'fda_earliest_year', 'fda_disease',
       'fda_documents', 'fda_disease<>drug', 'fda_AP'],
      dtype='object')

## Report translational stats

In [54]:
from typing import Optional, List


In [55]:
def _is_linked(e, mapping_lookup):
    if e not in mapping_lookup:
        print(f"{e}: not_found")
    mid = mapping_lookup.get(e, "-1")
    #if mid == "-1":
     #   print(f"not linked entity: {e}")
    return str(mid).strip() != "-1"

def _is_linked_id_based(e_drug_id, e_disease_id):
    return (str(e_drug_id).strip() != "-1") and (str(e_disease_id).strip() != "-1")

def _is_short(e, mapping_lookup):
    if len(e.strip()) < 3:
        print(f"{e}, {_is_linked(e,mapping_lookup)}")

    return len(e.strip()) < 3

In [65]:
def clinical_reach_report(
    preclinical_df: pd.DataFrame,
    filtered_df: pd.DataFrame,
    *,
    drugs_col_to_use: str,
    disease_col_to_use: str,
    mapping_lookup: dict = None,
    fda_preclin_not_clin: Optional[List[str]] = None,
    key_col: str = "normalized_key",
    phase4_col: str = "at_least_one_phase4",
    phase3_completed_col: str = "at_least_one_phase3_completed",
    fda_approval_col: str = "fda_AP",
    ignore_id: str = "-1",
    case_insensitive: bool = False,
    selected_disease: str = None,
):
    # optionally lowercase mapping keys once
    if case_insensitive:
        mapping_lookup = {str(k).lower(): v for k, v in mapping_lookup.items()}

    def fmt(n): return f"{int(n):,}"
    def pct(a, b): return (a / b * 100) if b else 0.0

    if selected_disease:
        print(f"Filtering datasets for disease: {selected_disease}")
        preclinical_df = preclinical_df[preclinical_df['merged_mondo_label'].str.contains(selected_disease, case=False, na=False)]
        filtered_df = filtered_df[filtered_df[key_col].str.contains(selected_disease, case=False, na=False)]
        fda_preclin_not_clin = [
            s for s in fda_preclin_not_clin
            if selected_disease.lower() in str(s).lower()
        ]
        print(f"Example diseases: {preclinical_df['merged_mondo_label'].dropna().unique()[:10]}")
        print(f"Example drugs: {preclinical_df['merged_umls_label'].dropna().unique()[:10]}")
        print(f"Example FDA: {fda_preclin_not_clin[:10]}")


    # --- denominators ---
    clinical_unique = set(filtered_df[key_col].dropna())
    preclin_unique = preclinical_df["disease<>drug"].dropna().unique()
    total_unique_preclin = len(preclin_unique)

    preclin_unique_drug = len(preclinical_df["merged_umls_label"].dropna().unique())
    preclin_unique_disease = len(preclinical_df["merged_mondo_label"].dropna().unique())

    preclin_series = preclinical_df["disease<>drug"].dropna()
    counts = preclin_series.value_counts()
    
    # preclinical drugs with count > 1
    preclin_gt1 = set(counts[counts > 1].index)
    
    # preclinical drugs with count == 1
    preclin_eq1 = set(counts[counts == 1].index)
    
    # ones that appear in clinical as well
    preclin_eq1_and_clin = preclin_eq1 & clinical_unique

    # excluded preclinical due to infrequent
    preclin_excl = preclin_eq1 - preclin_eq1_and_clin
    print(f"Example excluded infrequent pairs: {list(preclin_excl)[:10]}")

    # rule:
    # (>1 preclinical) OR (==1 preclinical AND appears in clinical at least once)
    total_unique_preclin_gt1 = len(preclin_gt1 | preclin_eq1_and_clin)

    preclin_unique_ids = list(set(zip(preclinical_df[drugs_col_to_use], preclinical_df[disease_col_to_use])))
    total_preclinical_linked = sum(
        _is_linked_id_based(e_drug_id, e_disease_id)
        for e_drug_id, e_disease_id in preclin_unique_ids
    )

    singletons = set(counts[counts == 1].index)
    singleton_mask = preclinical_df["disease<>drug"].isin(singletons)

    linked_mask = preclinical_df.apply(
        lambda r: _is_linked_id_based(r[drugs_col_to_use], r[disease_col_to_use]),
        axis=1
    )
    
    linked_singletons_df = preclinical_df[singleton_mask & linked_mask]

    # set of drugs that satisfy the gt1 rule
    preclin_gt1_rule = preclin_gt1 | (preclin_eq1 & clinical_unique)
    
    # linked subset of that
    eligible_mask = preclinical_df["disease<>drug"].isin(preclin_gt1_rule)
    preclinical_df_gt1 = preclinical_df.loc[eligible_mask]
    preclin_unique_ids_gt1 = list(set(zip(preclinical_df_gt1[drugs_col_to_use], preclinical_df_gt1[disease_col_to_use])))
    total_preclinical_linked_gt1 = sum(
        _is_linked_id_based(e_drug_id, e_disease_id)
        for e_drug_id, e_disease_id in preclin_unique_ids_gt1
    )
    # --- reached clinical ---
    
    # pairs either in CT.gov dataset, or with a link to FDA
    reached_clinical_entities = clinical_unique | fda_preclin_not_clin
    reached_clinical = len(clinical_unique) + len(fda_preclin_not_clin)

    counts = np.array(filtered_df["preclinical_count"])
    total_unique_reached_clin_gt1 = int((counts > 1).sum())

    clin_unique_ids = list(set(zip(filtered_df[drugs_col_to_use], filtered_df[disease_col_to_use])))
    reached_clinical_and_linked = sum(
        _is_linked_id_based(e_drug_id, e_disease_id)
        for e_drug_id, e_disease_id in clin_unique_ids
    )


    # --- phase 3 completed / phase 4 / FDA ---
    df_p3c_or_p4_or_fda = filtered_df[
        filtered_df[phase3_completed_col].fillna(False)
        | filtered_df[phase4_col].fillna(False)
        | filtered_df[fda_approval_col].fillna(False)
    ]

    p3c_or_p4_unique = df_p3c_or_p4_or_fda[key_col].dropna().unique()

    reached_p3c_or_p4_or_fda = len(p3c_or_p4_unique) + len(fda_preclin_not_clin)

    p3c_or_p4_unique_ids = list(set(zip(df_p3c_or_p4_or_fda[drugs_col_to_use], df_p3c_or_p4_or_fda[disease_col_to_use])))
    reached_p3c_or_p4_and_linked = sum(
        _is_linked_id_based(e_drug_id, e_disease_id)
        for e_drug_id, e_disease_id in p3c_or_p4_unique_ids
    )

    # --- print report ---
    print("\n" + "=" * 70)
    print("Clinical reach report (preclinical → clinical) [flat entities]")
    print("=" * 70)

    print("\nDenominators (preclinical diversity):")
    print(f"Note excl. infrequent rule: (>1 preclinical) OR \n (==1 preclinical AND >=1 clinical)")
    print(f"  Unique disease:                             {fmt(preclin_unique_disease)}")
    print(f"  Unique drug:                               {fmt(preclin_unique_drug)}")
    print(f"  Unique disease<>drug:                    {fmt(total_unique_preclin)}")
    
    print(f"  Unique disease<>drug excl. infrequent:                 {fmt(total_unique_preclin_gt1)}")
    print(f"  Unique disease<>drug successfully linked:               {fmt(total_preclinical_linked)}")
    print(f"  Unique disease<>drug linked excl. infrequent:           {fmt(total_preclinical_linked_gt1)}")
    print(f"  Unique disease<>drug from FDA:                  {fmt(len(fda_preclin_not_clin))}")

    print("\nReach metrics (unique drugs):")
    print(f"  Reached clinical (any / linked):        {fmt(reached_clinical)} /{fmt(reached_clinical_and_linked)}")
    print(f"    = {pct(reached_clinical, total_unique_preclin):.2f}% of all unique")
    print(f"    = {pct(reached_clinical, total_unique_preclin_gt1):.2f}% excl. only once in preclinical")
    print(f"    = {pct(reached_clinical_and_linked, total_preclinical_linked):.2f}% of linked unique")
    print(f"    = {pct(reached_clinical_and_linked, total_preclinical_linked_gt1):.2f}% of linked unique excl. entities only once in animal DS")


    print(f"\n  Reached (P3 completed / P4 / FDA) (any / linked):     {fmt(reached_p3c_or_p4_or_fda)} /  {fmt(reached_p3c_or_p4_and_linked)}")
    print(f"    = {pct(reached_p3c_or_p4_or_fda, total_unique_preclin):.2f}% of all unique")
    print(f"    = {pct(reached_p3c_or_p4_or_fda, total_unique_preclin_gt1):.2f}% excl. only once in preclinical")
    print(f"    = {pct(reached_p3c_or_p4_and_linked, total_preclinical_linked):.2f}% of linked unique")
    print(f"    = {pct(reached_p3c_or_p4_and_linked, total_preclinical_linked_gt1):.2f}% of linked unique excl. entities only once in animal DS")

    print("\n" + "-" * 70)
    print("Note: percentages are over unique entity types (not mention frequency).")
    print("-" * 70)

    return {
        "total_unique_preclin": total_unique_preclin,
        "total_unique_preclin_gt1": total_unique_preclin_gt1,
        "total_preclinical_linked": total_preclinical_linked,
        "reached_clinical": reached_clinical,
        "reached_clinical_and_linked": reached_clinical_and_linked,
        "reached_phase3_completed_or_phase4_or_fda": reached_p3c_or_p4_or_fda,
        "reached_phase3_completed_or_phase4_and_linked": reached_p3c_or_p4_and_linked,
    }, linked_singletons_df, reached_clinical_entities, p3c_or_p4_unique

In [57]:
15823/271230

0.05833794196807138

In [71]:
report, linked_singletons_df, reached_clinical_entities, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin)


Example excluded infrequent pairs: ['down syndrome <> atropine', 'pulmonary hypertension <> selective et (a) antagonist', 'obstructive lung disease <> jq1', 'oxygen deprivation-induced cell injury <> mp glycine', 'parkinson disease <> mglu receptor ligands', 'gliosis <> substance with interleukin 1 receptor antagonist mechanism of action', 'cancer <> tpcs2a', 'stanford type-a acute aortic dissection <> probdnf monoclonal antibody', 'multiple sclerosis <> ro9021', 'thermal hyperalgesia <> d-phe-cys-tyr-d-trp-arg-pen-thr-nh2']

Clinical reach report (preclinical → clinical) [flat entities]

Denominators (preclinical diversity):
Note excl. infrequent rule: (>1 preclinical) OR 
 (==1 preclinical AND >=1 clinical)
  Unique disease:                             79,857
  Unique drug:                               291,457
  Unique disease<>drug:                    1,808,059
  Unique disease<>drug excl. infrequent:                 271,230
  Unique disease<>drug successfully linked:              

## check overlaps with drugs only stats

In [72]:
drugs_from_pairs = {
    item.split("<>")[1].strip()
    for item in reached_clinical_entities
}


In [73]:
len(drugs_from_pairs)

3515

In [74]:
with open("out/reached_clinical_drug_entities.json", "r") as f:
    drugs_only = set(json.load(f))


In [75]:
len(drugs_only)

6221

In [76]:
drugs_not_translated_with_disease = drugs_only - drugs_from_pairs

In [77]:
len(drugs_not_translated_with_disease)

2706

In [78]:
drugs_not_translated_with_disease

{'ganoderma',
 'non-vitamin k antagonist oral anticoagulants',
 '1-(2-methoxy-phenyl)-4-(4-(4-(6-imidazol(2,1-b) thiazolyl)-phenoxy)-butyl-4-(14)c)-piperazine dimethane',
 'indium (111-in) pentetate',
 'bespar',
 'mitocurcumin',
 'btrx-246040',
 'mtp-131',
 'fish derived omega 3 fatty acid',
 'amoxicillin sodium',
 'disulfide, glutathione',
 'triflupromazine',
 'guanidinoacetate',
 'fruquintinib',
 'nitro-bid',
 'volatile',
 'cham',
 'guanethidine',
 'alpha2-receptor agonist',
 'itf2984',
 'sulfonamide',
 'brilliant blue g',
 'carbamoyltransferase, ornithine',
 'depo-medrol',
 'brompheniramine',
 'n-3 lcpufas',
 'staril',
 'potassium carbonate',
 'mineralocorticoid receptor',
 'fractionated',
 'ropivacaine hydrochloride',
 'ketogenic',
 'technetium tc 99m pertechnetate',
 'glycyrrhiza glabra extract',
 'alpha 1 antitrypsin',
 'follistim',
 'tpn',
 'bevacizumab-irdye800cw',
 'dexamethasone group',
 'flav',
 'technetium (99m-tc) exametazime',
 'dirithromycin',
 'abi',
 '4-(3-thioxo-3h-1,

In [100]:
preclinical_df[
    preclinical_df['merged_umls_label']
    .str.contains("cefiderocol", case=False, na=False)
]

,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid,pub_year,disease<>drug
587631,34223152,prosthetic joint infection,prosthesis-related infectious disease,MONDO:0043892,prosthesis-related infectious disease,-1,prosthesis-related infectious disease,MONDO:0043892,cefiderocol|siderophore-conjugated cephalospor...,cefiderocol|siderophore-conjugated cephalospor...,C4548369|-1|C1260298,-1,cefiderocol,C4548369,2021.0,prosthesis-related infectious disease <> cefid...
635538,31591126,maltophilia|s . maltophilia infections,maltophilia|s . maltophilia infections,-1|-1,maltophilia|s . maltophilia infections,-1,maltophilia,-1,ceftazidime|cefiderocol|siderophore cephalosporin,Ceftazidime|cefiderocol|siderophore cephalosporin,C0007559|C4548369|-1,-1,cefiderocol,C4548369,2019.0,maltophilia <> cefiderocol
635541,31591126,maltophilia|s . maltophilia infections,maltophilia|s . maltophilia infections,-1|-1,maltophilia|s . maltophilia infections,-1,s . maltophilia infections,-1,ceftazidime|cefiderocol|siderophore cephalosporin,Ceftazidime|cefiderocol|siderophore cephalosporin,C0007559|C4548369|-1,-1,cefiderocol,C4548369,2019.0,s . maltophilia infections <> cefiderocol
1260770,39452245,keratitis|aeruginosa keratitis|bacterial kerat...,keratitis|aeruginosa keratitis|corneal infecti...,MONDO:0003085|-1|MONDO:0023865|MONDO:0036501|-1,keratitis|aeruginosa keratitis|corneal infecti...,-1|-1|-1,keratitis,MONDO:0003085,ciprofloxacin|cefiderocol|tobramycin,Ciprofloxacin|cefiderocol|Tobramycin,C0008809|C4548369|C0040341,-1,cefiderocol,C4548369,2024.0,keratitis <> cefiderocol
1260773,39452245,keratitis|aeruginosa keratitis|bacterial kerat...,keratitis|aeruginosa keratitis|corneal infecti...,MONDO:0003085|-1|MONDO:0023865|MONDO:0036501|-1,keratitis|aeruginosa keratitis|corneal infecti...,-1|-1|-1,aeruginosa keratitis,-1,ciprofloxacin|cefiderocol|tobramycin,Ciprofloxacin|cefiderocol|Tobramycin,C0008809|C4548369|C0040341,-1,cefiderocol,C4548369,2024.0,aeruginosa keratitis <> cefiderocol
1260776,39452245,keratitis|aeruginosa keratitis|bacterial kerat...,keratitis|aeruginosa keratitis|corneal infecti...,MONDO:0003085|-1|MONDO:0023865|MONDO:0036501|-1,keratitis|aeruginosa keratitis|corneal infecti...,-1|-1|-1,corneal infection,MONDO:0023865,ciprofloxacin|cefiderocol|tobramycin,Ciprofloxacin|cefiderocol|Tobramycin,C0008809|C4548369|C0040341,-1,cefiderocol,C4548369,2024.0,corneal infection <> cefiderocol
1260779,39452245,keratitis|aeruginosa keratitis|bacterial kerat...,keratitis|aeruginosa keratitis|corneal infecti...,MONDO:0003085|-1|MONDO:0023865|MONDO:0036501|-1,keratitis|aeruginosa keratitis|corneal infecti...,-1|-1|-1,refractory malignant neoplasm,MONDO:0036501,ciprofloxacin|cefiderocol|tobramycin,Ciprofloxacin|cefiderocol|Tobramycin,C0008809|C4548369|C0040341,-1,cefiderocol,C4548369,2024.0,refractory malignant neoplasm <> cefiderocol
1260782,39452245,keratitis|aeruginosa keratitis|bacterial kerat...,keratitis|aeruginosa keratitis|corneal infecti...,MONDO:0003085|-1|MONDO:0023865|MONDO:0036501|-1,keratitis|aeruginosa keratitis|corneal infecti...,-1|-1|-1,extensively drug-resistant,-1,ciprofloxacin|cefiderocol|tobramycin,Ciprofloxacin|cefiderocol|Tobramycin,C0008809|C4548369|C0040341,-1,cefiderocol,C4548369,2024.0,extensively drug-resistant <> cefiderocol
1339758,36154614,stenotrophomonas maltophilia|s . maltophilia p...,stenotrophomonas maltophilia|s . maltophilia p...,-1|-1|MONDO:0005249|-1|-1,stenotrophomonas maltophilia|s . maltophilia p...,-1,stenotrophomonas maltophilia,-1,cefiderocol|trimethoprim-sulfamethoxazole|pare...,cefiderocol|Trimethoprim|parenteral siderophor...,C4548369|C0041041|-1,-1,cefiderocol,C4548369,2022.0,stenotrophomonas maltophilia <> cefiderocol
1339761,36154

In [101]:
clinical_df[
    clinical_df['merged_umls_label']
    .str.contains("cefiderocol", case=False, na=False)
]

,nct_id,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,drug_umls_term_norm,...,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid,disease<>drug,phase,overall_status,start_date,completion_date,trial_start_year,trial_completion_year


In [102]:
fda_drug_disease[
    fda_drug_disease['fda_merged_umls_label']
    .str.contains("cefiderocol", case=False, na=False)
]

,fda_merged_umls_label,fda_disease_mondo_term_norm,fda_earliest_year,fda_disease,fda_documents,fda_disease<>drug,fda_AP
8083,cefiderocol,bacterial pneumonia,2019.0,hospital-acquired bacterial pneumonia,['NDA209445'],bacterial pneumonia <> cefiderocol,True
8084,cefiderocol,complicated urinary tract infections (cuti),2019.0,complicated urinary tract infections (cuti),['NDA209445'],complicated urinary tract infections (cuti) <>...,True
8085,cefiderocol,pyelonephritis,2019.0,pyelonephritis,['NDA209445'],pyelonephritis <> cefiderocol,True
8086,cefiderocol,ventilator-associated bacterial pneumonia,2019.0,ventilator-associated bacterial pneumonia,['NDA209445'],ventilator-associated bacterial pneumonia <> c...,True


### check specific diseases

In [59]:
97/1026

0.09454191033138401

In [52]:
report, linked_singletons_df, clinical_unique, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin,
    selected_disease="stroke")

Filtering datasets for disease: stroke
Example diseases: ['stroke disorder' 'obsolete susceptibility to ischemic stroke'
 'focal stroke' 'thromboembolic stroke' 'stroke neuroinflammation'
 'photothrombotic stroke' 'neonatal stroke' 'chronic stroke'
 'pediatric arterial ischemic stroke' 'occlusive stroke']
Example drugs: ['hydroxymethylglutaryl coa reductase inhibitors' '3-methyladenine'
 'simvastatin' 'hsch2ch(ch2ch(ch3)2)co-phenylalanyl-alaninamide' 'xq-1 h'
 'clopidogrel'
 '10-o-(n,n-dimethylaminoethyl)ginkgolide b methanesulfonate' 'xq-1h'
 'jsh 23' 'qnz']
Example FDA: ['stroke disorder <> icosapent ethyl', 'stroke disorder <> dantrolene']

Clinical reach report (preclinical → clinical) [flat entities]

Denominators (preclinical diversity):
Note excl. infrequent rule: (>1 preclinical) OR 
 (==1 preclinical AND >=1 clinical)
  Unique disease:                             414
  Unique drug:                               12,883
  Unique disease<>drug:                    19,891
  Unique 

In [63]:
report, linked_singletons_df, clinical_unique, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin,
    selected_disease="alzheimer")

Filtering datasets for disease: alzheimer
Example diseases: ['alzheimer disease' 'familial alzheimer disease' "alzheimer's behavior"
 "diabetes-related alzheimer's disease"
 "sporadic mouse model of alzheimer's disease"
 "alzheimer's disease-related memory impairment" "alzheimer's therapy"
 "alzheimer's disease neuropathy" 'alzheimer disease neuropathy'
 'alzheimer-like spatial memory impairment']
Example drugs: ['phytol-plga' 'phytol' 'activator of sirtuin-1'
 'poly (adp-ribose) synthase-1' 'simct4' 'rivastigmine'
 'phosphatidylserine' 'microtubule-binding drug' 'combretastatin a4'
 'pnr502']
Example FDA: []
Example excluded infrequent pairs: ['alzheimer disease <> aggregation dual inhibitor', 'alzheimer disease <> glycogen synthase kinase-3 (gsk-3) inhibitor lithium chloride', 'alzheimer disease <> non-specific alpha (2) adrenergic receptor agonist', 'alzheimer disease <> kinase, protein', 'alzheimer disease <> and butyrylcholinesterase/monoamine oxidase a-b inhibitor', 'alzheimer di

In [60]:
report, linked_singletons_df, clinical_unique, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin,
    selected_disease="multiple sclerosis")

Filtering datasets for disease: multiple sclerosis
Example diseases: ['multiple sclerosis' 'relapsing-remitting multiple sclerosis'
 'primary progressive multiple sclerosis'
 'chronic progressive multiple sclerosis'
 'progressive relapsing multiple sclerosis'
 'secondary progressive multiple sclerosis'
 'multiple sclerosis, susceptibility to'
 'multiple sclerosis-associated central neuropathic pain'
 'multiple sclerosis-related spasticity' 'multiple sclerosis pathogenesis']
Example drugs: ['fumarate, dimethyl' 'fingolimod' '2cf34ap' 'dalfampridine'
 '3-trifluoromethylpyridine' '2-trifluoromethyl-4-ap' '3me4ap'
 '4-aminopyridine k [+] channel blockers' '3-methyl-4-aminopyridine'
 '3f4ap']
Example FDA: ['multiple sclerosis <> dantrolene']
Example excluded infrequent pairs: ['multiple sclerosis <> cd40 small interfering rna', 'multiple sclerosis <> tm5484', 'multiple sclerosis <> d60', 'secondary progressive multiple sclerosis <> s1pr agonists', 'multiple sclerosis <> docetaxel', 'multipl

In [64]:
report, linked_singletons_df, clinical_unique, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin,
    selected_disease="epilepsy")

Filtering datasets for disease: epilepsy
Example diseases: ['glioma-associated epilepsy' 'epilepsy' 'complex partial epilepsy'
 'epilepsy with generalized tonic-clonic seizures'
 'idiopathic generalized epilepsy' 'epilepsy syndrome'
 'obsolete absence epilepsy' 'temporal lobe epilepsy'
 'post-traumatic epilepsy' 'childhood absence epilepsy']
Example drugs: ['perampanel' 'ginsenoside m1' 'drugs' 'antiepileptic' 'levetiracetam'
 'ocimum sanctum whole extract' 'catalpol' 'mannitol' 'pyrilamine' 'e177']
Example FDA: ['focal epilepsy <> primidone', 'epilepsy <> dilantin', 'childhood absence epilepsy <> valproate', 'complex partial epilepsy <> carbamazepine', 'juvenile myoclonic epilepsy <> levetiracetam', 'complex partial epilepsy <> valproate sodium', 'focal epilepsy <> everolimus', 'complex partial epilepsy <> phenytoin', 'obsolete absence epilepsy <> divalproex sodium', 'childhood absence epilepsy <> methsuximide']
Example excluded infrequent pairs: ['epilepsy <> deslanoside', 'epilepsy 

In [65]:
report, linked_singletons_df, clinical_unique, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin,
    selected_disease="schizophrenia")

Filtering datasets for disease: schizophrenia
Example diseases: ['schizophrenia' 'cognitive impairment associated with schizophrenia'
 'negative symptoms of schizophrenia' 'obsolete schizophrenia'
 'paranoid schizophrenia' 'schizophrenia and bipolar disorder'
 'treatment-refractory schizophrenia' 'deletion-associated schizophrenia'
 "autism alzheimer's schizophrenia"
 'cognitive impairment in schizophrenia']
Example drugs: ['nicotine' 'ferulic acid' 'cholinesterase inhibitor' 'ferulate' 'fk-506'
 'tacrolimus' 'clozapine' 'haloperidol'
 'pyrrolidine allosteric modulators' 'aniracetam']
Example FDA: ['schizophrenia <> hydrochloride, chlorpromazine', 'paranoid schizophrenia <> fluphenazine', 'schizophrenia <> thioridazine']
Example excluded infrequent pairs: ['schizophrenia <> βarr2-biased d2r ligand', 'schizophrenia <> second generation atypical apd', 'schizophrenia <> agonist pam', 'schizophrenia <> eeds', 'schizophrenia <> no [•] releasers', 'schizophrenia <> muscarinic m1 receptor ago

In [66]:
report, linked_singletons_df, clinical_unique, p3c_or_p4_unique = clinical_reach_report(
    preclinical_df=preclinical_df,
    filtered_df=filtered_df,
    drugs_col_to_use="merged_umls_termid",
    disease_col_to_use="merged_mondo_termid",
    fda_preclin_not_clin=fda_preclin_not_clin,
    selected_disease="sleep")

Filtering datasets for disease: sleep
Example diseases: ['sleep apnea syndrome' 'complex sleep apnea'
 'obstructive sleep apnea syndrome' 'excessive daytime sleepiness'
 'sleep deprivation' 'high-altitude sleep disturbance'
 'sleep-wake disorder' 'sleep' 'sleep disorder'
 'sleep deprivation-induced pain hypersensitivity']
Example drugs: ['α1-adrenergic-antagonist' 'alpha-1 adrenergic-antagonist'
 'α1-adrenergic antagonists' 'α1-adrenergic antagonist'
 'protocatechuic acid' 'nonselective d2 antagonist' 'sch 39166'
 'modafinil' 'selective d1 antagonist' '91356a']
Example FDA: ['sleep disorder <> hydrochloride, doxepin']
Example excluded infrequent pairs: ['sleep apnea syndrome <> tetrakis(n-methyl-4-pyridiniumyl)porphine manganese(iii) complex', 'obstructive sleep apnea syndrome <> 5ht2', 'rem sleep behavior disorder <> beta adrenoceptor', 'sleep <> terazosin', 'obstructive sleep apnea syndrome <> u 0126', 'sleep-wake disorder <> dual orexin 1 and orexin 2 receptor antagonist', 'paradoxi

In [ ]:
clinical_unique

In [107]:
example = "multiple sclerosis <> ponesimod"
example in p3c_or_p4_unique

True

In [101]:
filtered_df[filtered_df['normalized_key']==example]

,normalized_key,clinical_doc_ids,merged_mondo_termid,merged_umls_termid,phase,overall_status,clinical_count,preclinical_doc_ids,merged_mondo_termid_right,merged_umls_termid_right,preclinical_count,max_phase,at_least_one_phase3,at_least_one_phase3_completed,at_least_one_phase4
48240,multiple sclerosis <> ponesimod,"[NCT01093326, NCT02425644, NCT02907177, NCT032...",MONDO:0005301,C2934701,"[Phase 2, Phase 3, Phase 3, Phase 3]","[Completed, Completed, Terminated, Completed]",4.0,"[31214480, 34986275, 36419359, 38243760]",MONDO:0005301,C2934701,4.0,Phase 3,True,True,False


In [ ]:
n_in_clinical_linked = sum(_is_linked(e, mapping_lookup) for e in list(filtered_df.normalized_key))
n_in_clinical_linked

In [92]:
13570/1808059, 9028/266688

(0.007505286055377617, 0.03385229181665467)

In [61]:
912+97

1009

In [62]:
97/1009

0.09613478691774034